<img src="https://hilpisch.com/tpq_logo_bic.png"
width="20%" align="right">

<a href="https://colab.research.google.com/github/yhilpisch/algocolab/blob/main/notebooks/00_colab_introduction.ipynb"
target="_blank"><img
src="https://colab.research.google.com/assets/colab-badge.svg"
alt="Open In Colab"/></a>


# Algorithmic Trading with Python & Google Colab

## Starter — Google Colab as a Quantitative Lab

### Drive, Packages, GPUs, and AI-Assisted Debugging

The Python Quants GmbH | https://tpq.io<br>
© Dr. Yves J. Hilpisch | https://hilpisch.com

This short companion notebook demonstrates why Colab is useful for the
webinar: Drive persistence, a ready scientific Python stack, optional GPU
acceleration, and an integrated Gemini debugging workflow.

Colab runtimes are temporary. Treat `/content` as scratch space and persist
anything important to Google Drive.


## 1. Connect the Runtime to Google Drive

Drive lets separate Colab runtimes share data, notebooks, checkpoints, and
experiment artifacts. The mount cell is intended for Colab; the local fallback
keeps this notebook inspectable outside Colab.


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount('/content/drive')  # connect persistent Google Drive storage
    RUNS_ROOT = Path('/content/drive/MyDrive/algo/runs')
else:
    RUNS_ROOT = Path('/Users/yves/Google Drive/My Drive/algo/runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)  # create the shared artifact root
print(f'Runtime: {"Colab" if IN_COLAB else "local"}')
print(f'Persistent artifacts: {RUNS_ROOT}')

## 2. Use the Scientific Python Stack Immediately

Colab commonly provides `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels`,
and `torch`. The exact versions can change, so record them in a serious run.


In [ ]:
import importlib
packages = ['numpy', 'pandas', 'matplotlib', 'scipy', 'statsmodels', 'torch']
for name in packages:
    module = importlib.import_module(name)  # import the available package
    print(f'{name}: {getattr(module, "__version__", "unknown")}')

## 3. A Non-Linear Function: OLS Features Versus a DNN

The known target is \(f(x)=0.35x+\sin(2x)\), observed with small random
noise. A linear OLS model recovers the trend but not the cycle. Polynomial OLS
adds non-linearity by hand; the DNN instead learns a non-linear mapping from
the raw input. This is a representation lesson, not a trading strategy.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
plt.style.use('seaborn-v0_8')  # Apply the shared plotting style
rng = np.random.default_rng(42)  # Fix the synthetic sample
x_train = rng.uniform(-3.0 * np.pi, 3.0 * np.pi, size=256)
x_grid = np.linspace(-3.0 * np.pi, 3.0 * np.pi, 600)
def target(values: np.ndarray) -> np.ndarray:
    return 0.35 * values + np.sin(2.0 * values)  # Trend plus cycle
y_train = target(x_train) + rng.normal(0.0, 0.12, size=len(x_train))
figure, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for axis, degree in zip(axes, (1, 3, 9)):
    transformer = PolynomialFeatures(degree, include_bias=False)
    model = LinearRegression().fit(
        transformer.fit_transform(x_train[:, None]),
        y_train,
    )  # Fit one polynomial OLS specification
    prediction = model.predict(transformer.transform(x_grid[:, None]))
    axis.scatter(x_train, y_train, s=9, alpha=0.35, label='sample')
    axis.plot(x_grid, target(x_grid), color='#002D5A', label='target')
    axis.plot(x_grid, prediction, color='#F2994A', label='OLS')
    axis.set(title=f'Polynomial degree {degree}', xlabel='x')
    axis.grid(alpha=0.35)
axes[0].set_ylabel('f(x)')
axes[0].legend(frameon=False)
plt.show()

In [ ]:
import torch
class FunctionDNN(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(1, 32),
            torch.nn.Tanh(),
            torch.nn.Linear(32, 32),
            torch.nn.Tanh(),
            torch.nn.Linear(32, 1),
        )  # Two hidden layers learn a smooth non-linear mapping
    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.network(values)  # Return one prediction per input
torch.manual_seed(42)  # Fix initial DNN weights
x_tensor = torch.tensor(x_train[:, None], dtype=torch.float32)
y_tensor = torch.tensor(y_train[:, None], dtype=torch.float32)
grid_tensor = torch.tensor(x_grid[:, None], dtype=torch.float32)
model = FunctionDNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = torch.nn.MSELoss()  # Minimize squared approximation error
snapshots = {}
for epoch in range(1, 1001):
    optimizer.zero_grad()  # Clear the previous gradient
    loss = loss_fn(model(x_tensor), y_tensor)  # Evaluate fitted values
    loss.backward()  # Differentiate the loss with respect to weights
    optimizer.step()  # Update the network weights
    if epoch in (100, 500, 1000):
        with torch.no_grad():
            snapshots[epoch] = model(grid_tensor).numpy().ravel()
figure, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for axis, epoch in zip(axes, (100, 500, 1000)):
    axis.scatter(x_train, y_train, s=9, alpha=0.35, label='sample')
    axis.plot(x_grid, target(x_grid), color='#002D5A', label='target')
    axis.plot(x_grid, snapshots[epoch], color='#F2994A', label='DNN')
    axis.set(title=f'DNN after {epoch:,} epochs', xlabel='x')
    axis.grid(alpha=0.35)
axes[0].set_ylabel('f(x)')
axes[0].legend(frameon=False)
plt.show()

### CPU/GPU Training Timing Uses a Larger Workload

The function-fitting plots use a small sample because they are easy to read.
The timing experiment uses the same architecture and target with 16,384 input
points and 30 full-batch epochs. It times training after data placement, so it
measures repeated model computation rather than one-time transfer overhead.


In [ ]:
import time
def synchronize(device: torch.device) -> None:
    if device.type == 'cuda':
        torch.cuda.synchronize()  # Finish queued GPU work before timing
def time_training(device_name: str) -> float:
    torch.manual_seed(42)  # Match the initial weights across devices
    device = torch.device(device_name)
    x = torch.linspace(-3.0 * np.pi, 3.0 * np.pi, 16_384)[:, None]
    y = 0.35 * x + torch.sin(2.0 * x)  # Create the same larger target
    model = FunctionDNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    x, y = x.to(device), y.to(device)  # Place fixed data before timing
    synchronize(device)
    started = time.perf_counter()
    for _ in range(30):
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
    synchronize(device)
    return time.perf_counter() - started
timings = {'cpu': time_training('cpu')}
if torch.cuda.is_available():
    timings['cuda'] = time_training('cuda')
for device_name, elapsed in timings.items():
    print(f'{device_name}: {elapsed:.3f} seconds')

## 4. Compare CPU and GPU Matrix Multiplication

Matrix multiplication is a deliberately simple proxy for tensor workloads.
The benchmark measures elapsed time on the available devices; it is not a
promise about end-to-end trading speed. Small matrices can be slower on a GPU
because transfer and startup overhead dominate.


In [ ]:
import time
import torch
size = 3 * 2048  # large enough to make GPU parallelism visible
device_names = ['cpu']
if torch.cuda.is_available():
    device_names.append('cuda')
for device_name in device_names:
    device = torch.device(device_name)
    left = torch.randn((size, size), device=device)  # allocate on the device
    right = torch.randn((size, size), device=device)
    if device.type == 'cuda':
        torch.cuda.synchronize()  # wait before starting the timed region
    started = time.perf_counter()
    product = left @ right  # run the matrix multiplication
    if device.type == 'cuda':
        torch.cuda.synchronize()  # include asynchronous GPU work
    elapsed = time.perf_counter() - started
    print(f'{device_name}: {elapsed:.3f} seconds')

## 5. A Financial Bug: Misaligning Signal and Return

A common backtest error uses today's return with today's signal. That gives
the strategy information from the period it is supposedly predicting. The
correct convention is: build the signal using information through time `t`,
then apply it to the return observed at `t+1`.


In [ ]:
import numpy as np
import pandas as pd
returns = pd.Series([0.01, -0.02, 0.03], name='return')
signal = np.sign(returns)  # uses today's return: forbidden information
wrong_pnl = signal * returns  # look-ahead-biased backtest result
correct_signal = signal.shift(1).fillna(0)  # trade after the signal exists
correct_pnl = correct_signal * returns  # apply yesterday's signal today
pd.DataFrame({'return': returns, 'signal': signal,
              'wrong_pnl': wrong_pnl,
              'correct_signal': correct_signal,
              'correct_pnl': correct_pnl})

### Ask Gemini to Diagnose the Bug

In Colab, select the suspicious cell and ask Gemini to explain the timing
convention. A useful prompt is:

> Audit this backtest for look-ahead bias. State the timestamp at which each
> signal is known, identify whether the position is shifted before applying the
> return, and rewrite the smallest correct code change. Do not assume that a
> profitable result is valid evidence.

Gemini can accelerate debugging, but the learner still verifies the timestamps,
the shifted arrays, and the economic interpretation.


## 6. Colab Niceties and Boundaries

- Use `%%time` or `time.perf_counter()` for quick measurements.
- Keep exploratory plots and tables close to the code that creates them.
- Save checkpoints and run manifests to Drive, not only `/content`.
- Pin or record package versions when results matter.
- Restart and rerun from the top before calling a notebook reproducible.
- GPU availability, session lifetime, and package versions can vary.

Colab is a convenient research lab—not a guarantee of persistent hardware or
production-grade execution.


---

<img src="https://hilpisch.com/tpq_logo_bic.png"
width="20%" align="right">
